# DevFlow - Lab: RAG Seletivo com Azure AI Search

**Curso:** Agentic Engineering - SkillGo
**Prof.:** Ives Santos

---

**RAG comum:** busca pelo texto da issue e manda os `k` primeiros para o LLM.

**RAG seletivo:** busca o que **a tarefa precisa** e **descarta** o que so passou perto.

| Metade | Pergunta | Como fazemos aqui |
|---|---|---|
| 1. Recuperar por intencao | *de onde* a tarefa precisa de contexto? | filtro por `categoria` no Azure, diferente para triagem e planejamento |
| 2. Podar | *quanto* do que veio vale o custo? | corta trechos com score muito abaixo do melhor |

Todo trecho enviado ao LLM custa tokens - e trecho irrelevante nao e neutro: dilui a atencao do
modelo e aumenta a chance de citacao errada.

## 1. Ambiente

Secrets do Colab:

| Segredo | Obrigatorio |
|---|---|
| `AZURE_SEARCH_ENDPOINT` | sim |
| `AZURE_SEARCH_INDEX` | sim (`devflow-kb`) |
| `AZURE_SEARCH_KEY` | sim (chave de consulta) |
| `ANTHROPIC_API_KEY` | so para a secao 6 |

In [ ]:
!pip install -q "azure-search-documents>=11.6" "langchain-anthropic>=1.0"

In [ ]:
from google.colab import userdata

def segredo(nome: str) -> str:
    try:
        return (userdata.get(nome) or "").strip()
    except Exception:
        return ""

ENDPOINT = segredo("AZURE_SEARCH_ENDPOINT")
INDICE = segredo("AZURE_SEARCH_INDEX") or "devflow-kb"
CHAVE = segredo("AZURE_SEARCH_KEY")
ANTHROPIC_API_KEY = segredo("ANTHROPIC_API_KEY")

assert ENDPOINT and CHAVE, "defina AZURE_SEARCH_ENDPOINT e AZURE_SEARCH_KEY nos Secrets"
print("indice:", INDICE, "| Claude:", "disponivel" if ANTHROPIC_API_KEY else "sem chave (secao 6 pulada)")

## 2. Classe de busca no Azure

A mesma busca do lab anterior, com um parametro a mais: `modo`.

- `hibrida` (texto + vetor): melhor para **encontrar**;
- `vetor`: o score e a similaridade entre pergunta e trecho (maior = mais parecido), o que o torna
  bom para **podar**. Na hibrida o score vem da fusao das duas listas e fica quase igual entre os
  resultados, entao nao serve para decidir o que cortar.

In [ ]:
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizableTextQuery


class BuscaAzure:
    def __init__(self, endpoint: str, indice: str, chave: str):
        self.cliente = SearchClient(endpoint, indice, AzureKeyCredential(chave))

    def buscar(self, pergunta: str, k: int = 4, filtro: str | None = None, modo: str = "hibrida") -> list[dict]:
        resultados = self.cliente.search(
            search_text=pergunta if modo == "hibrida" else None,
            vector_queries=[VectorizableTextQuery(text=pergunta, k_nearest_neighbors=max(k, 10), fields="content_vector")],
            select=["metadata_storage_name", "ordinal_position", "titulo", "content", "categoria"],
            filter="titulo ne ''" + (f" and {filtro}" if filtro else ""),
            top=k,
        )
        trechos = []
        for doc in resultados:
            documento = doc["metadata_storage_name"].removesuffix(".md")
            trechos.append({
                "id": f"{documento}#{doc['ordinal_position']}",
                "categoria": doc.get("categoria"),
                "titulo": doc["titulo"],
                "texto": doc["content"].strip(),
                "score": round(doc["@search.score"], 4),
            })
        return trechos


def mostrar(trechos: list[dict]) -> None:
    for t in trechos:
        print(f"   [{t['score']:.4f}] {t['categoria'] or '-':<12} {t['id']:<28} {t['titulo']}")


def tokens(trechos: list[dict]) -> int:
    """Estimativa grosseira: ~4 caracteres por token."""
    return sum(len(t["titulo"]) + len(t["texto"]) for t in trechos) // 4


busca = BuscaAzure(ENDPOINT, INDICE, CHAVE)

## 3. A issue

In [ ]:
ISSUE = {
    "id": "ISSUE-1042",
    "titulo": "Cupom de desconto e aplicado duas vezes quando o cliente volta para a etapa de pagamento",
    "descricao": (
        "Ao voltar de PAGAMENTO para ENDERECO e avancar de novo, o desconto de 10% aparece somado duas vezes "
        "e o total sai 20% menor. 37 pedidos afetados em 48 horas. Nos logs aparece uma segunda chamada de "
        "POST /pricing/quote sem cabecalho de idempotencia apos o evento cart.state_changed."
    ),
    "criterios_aceite": [
        "Aplicar o mesmo cupom mais de uma vez nao deve alterar o total alem do primeiro desconto",
        "Voltar e avancar entre ENDERECO e PAGAMENTO deve manter o total estavel",
        "Deve existir teste automatizado que reproduza o defeito e falhe antes da correcao",
        "A correcao deve poder ser desligada sem novo deploy",
    ],
}

CONSULTA = " ".join([ISSUE["titulo"], ISSUE["descricao"], *ISSUE["criterios_aceite"]])

## 4. RAG comum

Busca pelo texto da issue, pega os 6 primeiros. Repare: **a politica de engenharia aparece?** Quem
relata um bug escreve "o total sai 20% menor", nunca "severidade" ou "prioridade" - entao a regra que a
triagem precisa quase nunca e a mais parecida com a issue.

In [ ]:
comum = busca.buscar(CONSULTA, k=6)
mostrar(comum)

print("\npolitica no contexto?", any(t["categoria"] == "politica" for t in comum))
print("tokens ~", tokens(comum))

## 5. RAG seletivo

### 5.1 Intencao: cada tarefa busca nas suas categorias

| Tarefa | Precisa de | Categorias |
|---|---|---|
| triagem | regras de severidade/prioridade/esforco e incidentes parecidos | `politica`, `incidentes` |
| planejamento | como o sistema funciona, como testar, como liberar | `arquitetura`, `testes`, `runbook` |

A consulta continua sendo o texto da issue - o **filtro** garante de onde o trecho vem. Dentro da
politica, por exemplo, a busca traz a secao mais ligada a *esta* issue.

In [ ]:
INTENCAO = {
    "triagem": ["politica", "incidentes"],
    "planejamento": ["arquitetura", "testes", "runbook"],
}

### 5.2 Poda: fica o melhor de cada categoria, e so o que chega perto dele

Para cada categoria:

1. busca ate `k_por_categoria` trechos no modo `vetor`;
2. o **primeiro sempre fica** - a tarefa exige aquela categoria;
3. os demais so ficam se `score >= fracao * score_do_primeiro`.

`fracao` e um botao de calibragem: `0.95` corta quase tudo, `0.80` quase nao corta. Olhe os scores do
seu indice antes de escolher.

In [ ]:
def recuperar_seletivo(consulta: str, tarefa: str, k_por_categoria: int = 3, fracao: float = 0.9):
    mantidos, descartados = [], []
    for categoria in INTENCAO[tarefa]:
        achados = busca.buscar(consulta, k=k_por_categoria, filtro=f"categoria eq '{categoria}'", modo="vetor")
        if not achados:
            continue
        corte = achados[0]["score"] * fracao
        mantidos.append(achados[0])
        for t in achados[1:]:
            (mantidos if t["score"] >= corte else descartados).append(t)
    return mantidos, descartados


for tarefa in INTENCAO:
    mantidos, descartados = recuperar_seletivo(CONSULTA, tarefa)
    print(f">>> {tarefa.upper()}  (categorias: {', '.join(INTENCAO[tarefa])})")
    print(" mantidos:")
    mostrar(mantidos)
    print(" podados:")
    mostrar(descartados) if descartados else print("   nenhum")
    print(f" tokens ~ {tokens(mantidos)}\n")

### 5.3 Comparando

In [ ]:
contexto_triagem, _ = recuperar_seletivo(CONSULTA, "triagem")
contexto_plano, _ = recuperar_seletivo(CONSULTA, "planejamento")

print(f"{'':<24}{'trechos':>8}{'tokens~':>9}{'politica?':>11}")
for nome, ctx in [("comum (k=6)", comum), ("seletivo: triagem", contexto_triagem), ("seletivo: planejamento", contexto_plano)]:
    print(f"{nome:<24}{len(ctx):>8}{tokens(ctx):>9}{str(any(t['categoria'] == 'politica' for t in ctx)):>11}")

## 6. Usando o contexto seletivo na triagem

O contexto da triagem vai para o Claude, que devolve a classificacao no formato do contrato `Triagem`.
Em `fontes` ele precisa citar os IDs recebidos - e conferimos se citou so o que recebeu.

In [ ]:
from typing import List, Literal
from pydantic import BaseModel, Field


class Triagem(BaseModel):
    severidade: Literal["baixa", "media", "alta", "critica"]
    prioridade: Literal["P0", "P1", "P2", "P3"]
    esforco: Literal["XS", "S", "M", "L", "XL"]
    justificativa: str = Field(description="Por que essa classificacao, em ate 4 frases, citando a politica")
    fontes: List[str] = Field(description="IDs dos trechos do contexto usados, ex.: politica-de-engenharia#3")


def montar_prompt(issue: dict, contexto: list[dict]) -> str:
    trechos = "\n\n".join(f"[{t['id']}] {t['titulo']}\n{t['texto']}" for t in contexto)
    criterios = "\n".join(f"- {c}" for c in issue["criterios_aceite"])
    return (
        f"ISSUE {issue['id']}\n{issue['titulo']}\n\n{issue['descricao']}\n\nCriterios de aceite:\n{criterios}"
        f"\n\nCONTEXTO\n{trechos}\n\n"
        "Classifique a issue usando exclusivamente a politica do contexto."
    )


if ANTHROPIC_API_KEY:
    import os
    from langchain_anthropic import ChatAnthropic

    os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY
    triador = ChatAnthropic(model="claude-opus-5", max_tokens=4000).with_structured_output(Triagem)
    triagem = triador.invoke(montar_prompt(ISSUE, contexto_triagem))

    print(f"severidade {triagem.severidade} | {triagem.prioridade} | esforco {triagem.esforco}")
    print("justificativa:", triagem.justificativa)
    print("fontes:", triagem.fontes)

    enviados = {t["id"] for t in contexto_triagem}
    print("\nfontes inventadas:", [f for f in triagem.fontes if f not in enviados] or "nenhuma")
    print(f"trechos citados: {len(set(triagem.fontes) & enviados)} de {len(enviados)} enviados")
else:
    print("pulado: sem ANTHROPIC_API_KEY. O prompt que seria enviado:\n")
    print(montar_prompt(ISSUE, contexto_triagem))